When installing packages, you may see `pip` or Colab complain about dependency conflicts. These shouldn't cause any problems.

In [1]:
%pip install cobalt-ai transformer_lens sae_lens

In [2]:
import json
from urllib.request import urlretrieve
import sae_lens
import numpy as np
import mapper
from mapper.filter_function_impl import gaussian_nbrs_density
import cobalt.multires_graph
from cobalt.coloring.color_spec import NumericColoringSpec
import ipyvuetify as v
import ipywidgets as w
import pandas as pd

In [3]:
if cobalt.config.is_colab_environment():
    from google.colab import output
    output.enable_custom_widget_manager()

We'll first use `sae_lens` to load an SAE. This one is a top-k SAE trained on the residual stream at layer 8 of GPT2-small. It could be easily replaced with any other supported SAE, although to get the most benefit it should be one with metadata available in Neuronpedia (particularly with explanations available to download [here](https://neuronpedia-exports.s3.amazonaws.com/index.html?prefix=explanations-only/)).

In [4]:
sae, cfg_dict, sparsity = sae_lens.SAE.from_pretrained_with_cfg_and_sparsity(
    release="gpt2-small-resid-post-v5-32k",
    sae_id="blocks.8.hook_resid_post",
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


cfg.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

sae_weights.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]

sparsity.safetensors:   0%|          | 0.00/131k [00:00<?, ?B/s]

In [5]:
# autodescriptions of features hosted by Neuronpedia
# This is used by the UI, but you could just use placeholder strings for a model without explanations available
explanations_path = "https://neuronpedia-exports.s3.amazonaws.com/explanations-only/gpt2-small_8-res_post_32k-oai.json"
urlretrieve(explanations_path, "explanations.json")
with open("explanations.json") as f:
    explanations = json.load(f)
explanations_dict = {
    int(exp["index"]): exp["description"] for exp in explanations
}

Now we'll build a TDA graph on the features using the SAE decoder weight vectors. This is a multiresolution graph, consisting of a series of "levels", each of which is a graph representing the set of features. The nodes of each graph are clusters of similar features, and there are weighted edges between similar clusters. The clusters are hierarchical between levels of the graph, meaning that each node in level `i` is a subset of a node at level `i+1`.

In [ ]:
# We will build a TDA graph using the SAE decoder weight vectors to represent each feature
# We could also use activation patterns of each feature on a large corpus (this is what we originally did)
# This gives a different perspective on feature geometry, but is more expensive to compute
feature_vecs = sae.W_dec.detach().numpy()

# There are a lot of hyperparameters we could tweak for this graph, but the defaults are good enough for now
g = mapper.quick_graph(feature_vecs, metric="cosine")

In [ ]:
np.save('feature_vecs.npy', feature_vecs)

You can access the individual graphs in `g` with the `levels` attribute. This is a list of graph objects, each of which has a `nodes` and an `edge_list` attribute.

This defines a custom UI for exploring the features in the graph. Shout out to Neuronpedia for providing super-convenient embeddable feature information views.

In [ ]:
class FeatureExplorer:
    def __init__(
        self,
        data_graph: mapper.HierarchicalPartitionGraph,
        feature_metadata: list[dict],
        feature_ids: list[int],
    ):
        self.data_graph = data_graph
        self.feature_metadata = feature_metadata
        self.displayed_feats = feature_ids
        self.global_feats_to_displayed_feats = {
            global_id: idx for idx, global_id in enumerate(feature_ids)
        }
        self.feature_description_strings = [self.feature_id_to_str(i) for i in feature_ids]

        feature_options = self.feature_description_strings.copy()
        initial_option = feature_options[0]
        self.feature_select = v.Autocomplete(
            dense=True,
            outlined=True,
            multiple=True,
            clearable=True,
            hide_details=True,
            class_="my-2",
            return_object=False,
            label="Select features",
            items=feature_options,
            v_model=[initial_option],
        )
        self.feature_select.on_event("change", self.select_feature_dropdown_cb)

        self.feature_detail_select = v.Autocomplete(
            dense=True,
            outlined=True,
            multiple=False,
            clearable=False,
            hide_details=True,
            class_="my-2",
            return_object=False,
            label="View feature detail",
            items=[initial_option],
            v_model=initial_option,
        )
        self.feature_detail_select.on_event(
            "change", self.update_displayed_feature_detail
        )
        self.feature_embed_widget = w.HTML("")

        self.graph_panel = v.Flex(children=[])
        self.feature_list = v.Flex(
            children=[self.feature_detail_select, self.feature_embed_widget]
        )
        self.update_displayed_feature_detail()
        self.layout = w.VBox([self.feature_select, self.graph_panel])

        self.displayed_landscape = cobalt.multires_graph.MultiResolutionLandscape(
            data_graph, init_max_nodes=1000, init_max_degree=4, use_rich_labels=True
        )
        self.displayed_landscape.set_coloring_spec(
            cobalt.multires_graph.NoColoringSpec()
        )
        self.graph_panel.children = [
            v.Flex(
                children=[
                    self.displayed_landscape,
                    v.Flex(children=[self.displayed_landscape.legend_box], style_="min-width:120px; padding:10px"),
                    self.feature_list,
                ],
                style_="display: flex;",
            )
        ]
        self.displayed_landscape.graph_viewer.on_select_points(
            self.graph_select_listener
        )

        nbr_graph = data_graph.neighbor_graph.raw_graph
        self.density = gaussian_nbrs_density(nbr_graph._neighbors, nbr_graph._distances)
        self.set_graph_coloring(pd.Series(self.density, name="density"), translate_indices=False)

    def node_labeler(self, u, f, node_val):
        feature_ids = [self.displayed_feats[i] for i in u]
        feature_names = [str(i) for i in feature_ids]
        feature_name_str = ", ".join(feature_names)

        return f"[{feature_name_str}]: {node_val:.3g}"

    def html_node_labeler(self, u, f, node_val):
        label_rows = (
            f"<div style='color:#222'>{self.feature_description_strings[i]}: {f[i]:.3g}</div>" for i in u
        )

        return {"content": "\n".join(label_rows)}

    def set_graph_coloring(self, f: pd.Series, translate_indices=True, **kwargs):
        if translate_indices:
            f = f.iloc[self.displayed_feats].reset_index(drop=True)

        coloring_spec = NumericColoringSpec(
            f,
            color_map_name="cividis",
            node_label_gen=self.node_labeler,
            rich_node_label_gen=self.html_node_labeler,
            rich_node_label_template="<div>${content}</div>",
            **kwargs
        )
        self.displayed_landscape.set_coloring_spec(coloring_spec)

    def show_features(self, feat_idxs: list[int], translate: bool = True):
        if not feat_idxs:
            return
        if translate:
            feat_idxs = [self.displayed_feats[i] for i in feat_idxs]
        feature_strs = [self.feature_id_to_str(i) for i in feat_idxs]
        self.feature_detail_select.items = feature_strs
        self.feature_detail_select.v_model = feature_strs[0]
        self.update_displayed_feature_detail()

    @staticmethod
    def get_feature_id_from_str(feat_str: str) -> int:
        split = feat_str.split(" ")
        return int(split[0])

    def feature_id_to_str(self, feat_id: int) -> str:
        desc = self.feature_metadata.get(feat_id, "")
        if desc:
            return f"{feat_id} ({desc.strip()})"
        else:
            return f"{feat_id} (no description)"

    def update_displayed_feature_detail(self, *_):
        feat_id = self.get_feature_id_from_str(self.feature_detail_select.v_model)
        self.feature_embed_widget.value = self.get_feature_embed_html(feat_id)

    def graph_select_listener(self, selection: list[int]):
        if selection:
            self.show_features(selection)

    def get_feature_embed_html(self, feature_idx: int):
        return (
            f"<iframe src=https://www.neuronpedia.org/gpt2-small/8-res_post_32k-oai/{feature_idx}"
            '?embed=true&embedexplanation=true&embedplots=true&embedtest=true" title="Neuronpedia" '
            'style="height: 100%; min-height: 500px; width: 100%; min-width: 600px"></iframe>'
        )

    def select_feature_dropdown_cb(self, *_):
        selected_features = self.feature_select.v_model
        selected_feature_ids = [int(f.split(" ")[0]) for f in selected_features]
        local_feature_ids = [
            self.global_feats_to_displayed_feats[i]
            for i in selected_feature_ids
            if i in self.global_feats_to_displayed_feats
        ]
        self.select_features(local_feature_ids)
        self.show_features(selected_feature_ids, translate=False)

    def select_features(self, feat_idxs, propagate: bool = False):
        if not isinstance(feat_idxs, list):
            feat_idxs = [feat_idxs]
        with self.displayed_landscape.graph_viewer.use_selection_listeners(propagate):
            self.displayed_landscape.graph_viewer.selected_points = feat_idxs
    @property
    def selected_feature_ids(self):
        selected_local_ids = self.displayed_landscape.graph_viewer.selected_points
        selected_global_ids = [self.displayed_feats[i] for i in selected_local_ids]
        return selected_global_ids

    def _ipython_display_(self):
        if cobalt.config.is_colab_environment():
            # workaround for https://github.com/googlecolab/colabtools/issues/3501
            from ipyvue import Html

            display(Html(tag="div", style_="display: none"), self.layout)
        else:
            display(self.layout)

In [ ]:
fe = FeatureExplorer(g, explanations_dict, list(range(feature_vecs.shape[0])))

Drag the "coarseness" slider (try a value a few notches lower than the starting value, maybe 14) to choose which level of the graph is displayed (i.e. how many features are in each node on average). and the "connectivity" slider (try lowering it to somewhere around 2) to choose the threshold for how strong an edge must be in order to be displayed (the number shown is the average degree of nodes in the displayed graph). Setting the connectivity to a very low level can help focus on a highly connected "backbone" of the feature space, while setting it to a high level can give a more holistic view of how everything fits together. Note that lowering the connectivity often produces a number of small components; you can click the "Reset Layout" button to group these all together and help focus on the main component of the graph.

You can select/deselect a node by double-clicking, or select a group with the lasso selection tools (hold the Alt key to activate the lasso tool more conveniently). When you select a node or set of nodes, the corresponding features will be added to the dropdown menu at the right. Selecting a feature from this menu will show the Neuronpedia feature summary.

You can also select one or more features from the dropdown menu at the top, and the corresponding nodes will be highlighted in the graph. This dropdown also allows searching in the feature descriptions.

If you want the graph to be larger, click the "Expand" button to make it take up all available horizontal space.

Note: Most of the interactive functionality requires the Colab kernel to be running, so things will seem strangely broken if you haven't run the notebook.

In [ ]:
fe

In [ ]:
# To highlight the nodes containing a set of features in the graph, you can use the menu at the top, or call this function:
fe.select_features([1234,2343], propagate=True)

In [ ]:
# If you have an array with one value for every feature, you can use it to set the node colors
fe.set_graph_coloring(pd.Series(sparsity.numpy(), name="log(act freq)"))

In [ ]:
np.save('sparsity.npy', sparsity.numpy())

In [ ]:
# If you select nodes in the graph, you can access the corresponding feature ids here
fe.selected_feature_ids

It can also be useful to build a graph on a subset of the features, for instance by selecting a group in the full graph to get more detail, or by filtering based on some other attribute. Here we'll build a graph on the top 10% most frequently activating features.

In [ ]:
sparsity_sort = np.argsort(sparsity.numpy())
top_sparsity_percentile = 0.1
n_points = int(top_sparsity_percentile * sparsity.shape[0])
top_sparsity_points = sparsity_sort[-n_points:]

In [ ]:
sparse_g = mapper.quick_graph(feature_vecs[top_sparsity_points, :], metric="cosine")

In [ ]:
fe = FeatureExplorer(sparse_g, explanations_dict, top_sparsity_points.tolist())

In [ ]:
fe